# Orientador — Clase 10: Archivos de texto y archivos CSV

**Asignatura:** Programación I / Algoritmos y Estructuras I  
**Docente:** Ing. María Eugenia Varando  

---

## 1. Concepto y organización de un archivo

* **a) ¿Qué significa que la información persista en el tiempo?:**  
  Significa que los datos quedan guardados físicamente en un soporte no volátil (disco, memoria flash, etc.) y no se pierden al finalizar la ejecución del programa ni al apagar la computadora, quedando disponibles para ejecuciones futuras.

* **b) ¿Qué diferencias encuentran entre una variable en memoria y un archivo?:**  
  * **Variable en memoria (RAM):** Es efímera y volátil; vive únicamente mientras el script está corriendo y se borra al cerrar el proceso.  
  * **Archivo:** Reside en almacenamiento permanente, permite que la información sobreviva a lo largo del tiempo y se procesa secuencialmente registro a registro para no saturar la memoria RAM.

* **c) ¿Qué ventajas y limitaciones presenta almacenar los datos como texto?:**  
  * **Ventajas:** Es portable y legible directamente por personas; puede abrirse, inspeccionarse o corregirse con cualquier editor simple (como el Bloc de notas).  
  * **Limitaciones:** Ocupa más espacio de almacenamiento que un archivo binario y requiere convertir los tipos de datos al leerlos (por ejemplo, transformar cadenas numéricas con `int()` o `float()`).

---

## 2. Apertura, procesamiento y cierre

### Análisis de los modos de apertura más frecuentes:
* **`"rt"` (Lectura):** Abre el archivo para leer texto. Requiere obligatoriamente que el archivo exista previamente; si no se encuentra en la ruta, se genera un error `OSError`.
* **`"wt"` (Escritura):** Abre el archivo para escribir texto. Si no existe lo crea en blanco; si ya existía, vacía o sobreescribe por completo todo su contenido anterior.
* **`"at"` (Agregado):** Abre el archivo para añadir texto. Si no existe lo crea, y si ya contiene información ubica el cursor al final para escribir nuevos registros sin borrar lo previo.

### Función de los componentes:
* **`open()`:** Establece el canal de comunicación con el sistema operativo para acceder al archivo según el modo indicado y retorna el manejador del archivo.
* **`write()`:** Graba una cadena de caracteres en el archivo. No agrega saltos de línea automáticamente, por lo que es necesario concatenar de forma explícita el carácter `\n`.
* **Estructura `with`:** Administra el contexto de forma segura garantizando que el archivo se cierre automáticamente al finalizar el bloque o ante cualquier excepción que ocurra dentro de él.

***
### Ejercicio 2 (Python):

In [1]:
def main():
    try:
        with open("notas.txt", "wt", encoding="utf-8") as arch:
            arch.write("Primer registro\n")
            arch.write("Segundo registro\n")
            arch.write("Tercer registro\n")
        print("Archivo notas.txt creado.")
    except OSError as err:
        print("Error de archivo:", err)


main()

Archivo notas.txt creado.


---

## 3. Escritura de registros de texto

Desarrollo de la función `crearAlumnos()` para generar `alumnos.txt`:
* Formato por renglón: `legajo;nombre;comision\n`
* Corte de carga con legajo igual a `-1` sin usar `break`.
* Validaciones: legajo positivo, nombre no vacío y sin el delimitador `;`, y comisión de `1` a `10`.
* Manejo de errores con `try-except` para valores numéricos (`ValueError`) y problemas de archivo (`OSError`).

***
### Ejercicio 3 (Python):

In [2]:
def pedirEntero(mensaje):
    valido = False
    num = 0
    while not valido:
        cadena = input(mensaje).strip()
        try:
            num = int(cadena)
            valido = True
        except ValueError:
            print("Debe ingresar un número entero.")
    return num


def crearAlumnos(modo="wt"):
    try:
        with open("alumnos.txt", modo, encoding="utf-8") as arch:
            legajo = pedirEntero("Ingrese legajo (-1 para salir): ")
            while legajo != -1:
                if legajo <= 0:
                    print("El legajo debe ser positivo.")
                else:
                    # Validamos que el nombre no este vacio ni contenga ';'
                    nombreValido = False
                    nombre = ""
                    while not nombreValido:
                        nombre = input("Nombre y apellido: ").strip()
                        if nombre == "":
                            print("El nombre no puede estar vacío.")
                        elif ";" in nombre:
                            print("El nombre no puede contener el carácter ';'.")
                        else:
                            nombreValido = True

                    comision = pedirEntero("Comisión (1 a 10): ")
                    while comision < 1 or comision > 10:
                        print("Comisión inválida. Debe ser entre 1 y 10.")
                        comision = pedirEntero("Comisión (1 a 10): ")

                    arch.write(str(legajo) + ";" + nombre + ";" + str(comision) + "\n")
                    print("Alumno guardado.")

                legajo = pedirEntero("Ingrese legajo (-1 para salir): ")

        print(f"Carga en modo {modo} finalizada.")
    except OSError as err:
        print("Error al acceder al archivo:", err)


def main():
    print("Carga inicial en modo wt:")
    crearAlumnos("wt")

    print("Reejecución en modo wt (sobrescribe):")
    crearAlumnos("wt")

    print("Carga en modo at (agrega al final):")
    crearAlumnos("at")


main()

Carga inicial en modo wt:
Alumno guardado.
Debe ingresar un número entero.
Carga en modo wt finalizada.
Reejecución en modo wt (sobrescribe):
El nombre no puede contener el carácter ';'.
Alumno guardado.
Alumno guardado.
Carga en modo wt finalizada.
Carga en modo at (agrega al final):
Alumno guardado.
Carga en modo at finalizada.


### Comparación entre modos `"wt"` y `"at"`:
* **Reejecución con `"wt"`:** Borra todo el contenido que tenía `alumnos.txt` y deja únicamente los datos que se ingresan en la nueva ejecución.
* **Ejecución con `"at"`:** Mantiene los registros anteriores y agrega los nuevos al final del archivo.

---

## 4. Lectura y procesamiento secuencial

Desarrollo de `mostrarAlumnos()` para recorrer `alumnos.txt` línea por línea con `for...in`:
* Quita del salto de línea con `rstrip("\n")`.
* Control de que cada línea contenga exactamente `3` campos con `split(";")` antes de convertir tipos.
* Muestra de registros válidos, filtro de legajos mayores a `10000` y acumulación por comisión en un diccionario.
* Reporte de registros con formato incorrecto sin interrumpir el procesamiento.

***
### Ejercicio 4 (Python):

In [3]:
def mostrarAlumnos():
    comisiones = {}
    errores = []

    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as arch:
            print("Alumnos válidos:")
            for linea in arch:
                lineaLimpia = linea.rstrip("\n")
                campos = lineaLimpia.split(";")

                if len(campos) == 3:
                    try:
                        legajo = int(campos[0].strip())
                        nombre = campos[1].strip()
                        comision = int(campos[2].strip())

                        if legajo > 0 and nombre != "" and 1 <= comision <= 10:
                            # a) Mostrar registro valido
                            print(f"Legajo: {legajo} - Alumno: {nombre} - Comisión: {comision}")

                            # b) Filtrar legajos mayores a 10000
                            if legajo > 10000:
                                print(f"Legajo mayor a 10000: {nombre} ({legajo})")

                            # c) Contar alumnos por comision en diccionario
                            comisiones[comision] = comisiones.get(comision, 0) + 1
                        else:
                            errores.append(lineaLimpia)
                    except ValueError:
                        errores.append(lineaLimpia)
                else:
                    errores.append(lineaLimpia)

        # c) Mostrar diccionario de comisiones
        print("Cantidad de alumnos por comisión:")
        for com in sorted(comisiones.keys()):
            print(f"Comisión {com}: {comisiones[com]}")

        # d) Mostrar lineas invalidas
        if len(errores) > 0:
            print("Registros con formato inválido:")
            for e in errores:
                print("Línea omitida:", e)
        else:
            print("No se registraron líneas con error.")

    except OSError as err:
        print("Error al leer el archivo:", err)


def main():
    mostrarAlumnos()


main()

Alumnos válidos:
Legajo: 9500 - Alumno: Ana Gomez - Comisión: 4
Legajo: 12300 - Alumno: Carlos Lopez - Comisión: 2
Legajo mayor a 10000: Carlos Lopez (12300)
Legajo: 15000 - Alumno: Lucia Diaz - Comisión: 4
Legajo mayor a 10000: Lucia Diaz (15000)
Cantidad de alumnos por comisión:
Comisión 2: 1
Comisión 4: 2
No se registraron líneas con error.


---

## 5. Archivos CSV y encabezados

Creación y lectura de `productos.csv` delimitado por `;`, sin utilizar el módulo `csv` de Python:
* Carga secuencial hasta ingresar `"FIN"` en el código, validando que código y descripción no estén vacíos ni contengan `;`.
* Lectura omitiendo la fila de encabezado `codigo;descripcion;cantidad;precio`.
* En `procesarProductos()`, se controla que código y descripción no estén vacíos, que la cantidad sea mayor o igual a `0` y que el precio sea un valor numérico finito mayor a `0`.
* Cálculo del subtotal por producto, acumulación del total general y advertencia si el stock es menor a `5`.

***
### Ejercicio 5 (Python):

In [2]:
def esFlotanteFinito(valor):
    return valor != float("inf") and valor != float("-inf") and valor == valor


def pedirCantidad():
    valido = False
    cant = 0
    while not valido:
        texto = input("Cantidad: ").strip()
        try:
            cant = int(texto)
            if cant >= 0:
                valido = True
            else:
                print("La cantidad debe ser mayor o igual a 0.")
        except ValueError:
            print("Error: ingrese un número entero válido para la cantidad.")
    return cant


def pedirPrecio():
    valido = False
    precio = 0.0
    while not valido:
        texto = input("Precio unitario: ").strip()
        try:
            precio = float(texto)
            if precio > 0 and esFlotanteFinito(precio):
                valido = True
            else:
                print("El precio debe ser un número finito mayor a 0.")
        except ValueError:
            print("Error: ingrese un valor numérico válido para el precio.")
    return precio


def cargarProductos():
    try:
        with open("productos.csv", "wt", encoding="utf-8") as arch:
            arch.write("codigo;descripcion;cantidad;precio\n")

            codigo = input("Código de producto (FIN para salir): ").strip()
            while codigo.upper() != "FIN":
                if codigo == "" or ";" in codigo:
                    print("El código no puede estar vacío ni contener ';'.")
                else:
                    descripcion = input("Descripción: ").strip()
                    while descripcion == "" or ";" in descripcion:
                        print("La descripción no puede estar vacía ni contener ';'.")
                        descripcion = input("Descripción: ").strip()

                    cantidad = pedirCantidad()
                    precio = pedirPrecio()

                    arch.write(f"{codigo};{descripcion};{cantidad};{precio}\n")
                    print("Producto registrado correctamente.")

                codigo = input("Código de producto (FIN para salir): ").strip()

    except OSError as err:
        print("Error al escribir el archivo:", err)


def procesarProductos():
    total = 0.0
    esCabecera = True

    try:
        with open("productos.csv", "rt", encoding="utf-8") as arch:
            print("Listado de productos:")
            for linea in arch:
                if esCabecera:
                    esCabecera = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) == 4:
                        cod = campos[0].strip()
                        desc = campos[1].strip()

                        # Se verifica que el codigo y la descripcion no esten vacios
                        if cod != "" and desc != "":
                            try:
                                cant = int(campos[2].strip())
                                precio = float(campos[3].strip())

                                # Se verifica cantidad >= 0 y precio positivo finito
                                if cant >= 0 and precio > 0 and esFlotanteFinito(precio):
                                    subtotal = cant * precio
                                    total += subtotal
                                    print(f"{cod} - {desc} | Stock: {cant} | Precio: ${precio} | Subtotal: ${subtotal}")

                                    if cant < 5:
                                        print(f"Alerta: stock bajo para {desc} ({cant} unidades)")
                                else:
                                    print("Línea con cantidad o precio fuera de rango ignorada.")
                            except ValueError:
                                print("Línea con datos numéricos inválidos ignorada.")
                        else:
                            print("Línea con código o descripción vacía ignorada.")
                    else:
                        print("Línea con cantidad incorrecta de campos ignorada.")

            print(f"Total general de stock: ${total}")

    except OSError as err:
        print("Error al acceder al archivo:", err)


def main():
    cargarProductos()
    procesarProductos()


main()

Producto registrado correctamente.
Producto registrado correctamente.
Producto registrado correctamente.
Listado de productos:
P01 - Teclado Mecanico | Stock: 10 | Precio: $20000.0 | Subtotal: $200000.0
P02 - Mouse Gamer | Stock: 5 | Precio: $18000.0 | Subtotal: $90000.0
P03 - Monitor LED | Stock: 20 | Precio: $100000.0 | Subtotal: $2000000.0
Total general de stock: $2290000.0


---

## 6. Manejo de excepciones

### Comparación entre `with` y `try-except-finally`

* **`with`:** Administra la apertura y cierra el archivo automáticamente al finalizar el bloque o ante cualquier excepción, evitando fugas de memoria o bloqueos.
* **`try-except-finally`:** Requiere declarar la variable del archivo en `None` previamente y verificar con `if arch is not None` antes de llamar a `close()`, ya que si `open()` falla la variable no queda ligada a ningún recurso.

***
### Ejercicio 6 (Python):

In [3]:
def esNumeroFinito(valor):
    return valor != float("inf") and valor != float("-inf") and valor == valor


def procesarArchivoSeguro(ruta):
    try:
        with open(ruta, "rt", encoding="utf-8") as arch:
            esEncabezado = True
            for linea in arch:
                if esEncabezado:
                    esEncabezado = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) != 4:
                        print("Fila con cantidad incorrecta de campos ignorada.")
                    else:
                        cod = campos[0].strip()
                        desc = campos[1].strip()
                        if cod != "" and desc != "":
                            try:
                                cant = int(campos[2].strip())
                                precio = float(campos[3].strip())

                                # Validamos que la cantidad sea >= 0 y el precio positivo y finito
                                if cant >= 0 and precio > 0 and esNumeroFinito(precio):
                                    print(f"{desc}: {cant} unidades a ${precio}")
                                else:
                                    print("Fila ignorada: la cantidad debe ser >= 0 y el precio positivo.")
                            except ValueError:
                                print("Fila con datos numéricos no válidos ignorada.")
                        else:
                            print("Fila con código o descripción vacía ignorada.")

    except OSError as err:
        print("No se pudo acceder al archivo:", err)


def main():
    print("Lectura de archivo existente:")
    procesarArchivoSeguro("productos.csv")

    print("Prueba de apertura con archivo inexistente:")
    procesarArchivoSeguro("archivo_inexistente.csv")


main()

Lectura de archivo existente:
Teclado Mecanico: 10 unidades a $20000.0
Mouse Gamer: 5 unidades a $18000.0
Monitor LED: 20 unidades a $100000.0
Prueba de apertura con archivo inexistente:
No se pudo acceder al archivo: [Errno 2] No such file or directory: 'archivo_inexistente.csv'


---

## 7. Desafío aplicado al Proyecto Integrador

Gestión de entregas mediante `entregas.csv` con la estructura `numero_grupo;nombre_proyecto;integrante_responsable;porcentaje_avance`:
* En `crearEntregasCsv()`, se comprueba si el archivo ya existe antes de crearlo para conservar las entregas anteriores y no sobrescribirlas con `"wt"`.
* En `grupoYaExiste()`, si ocurre un `OSError` al abrir el archivo se relanza la excepción con `raise` para no interpretar un fallo de lectura como si el grupo no estuviera registrado.
* En `mostrarEntregas()`, `buscarEntrega()`, `proyectosAvanceMenor50()` y `calcularPromedioAvance()` se validan todos los campos del registro antes de procesarlo.

***
### Ejercicio 7 (Python):

In [4]:
def esAvanceFinito(valor):
    return valor != float("inf") and valor != float("-inf") and valor == valor


# a) Crear archivo y escribir encabezado sin pisar entregas anteriores
def crearEntregasCsv():
    archivoExiste = False
    try:
        with open("entregas.csv", "rt", encoding="utf-8") as arch:
            archivoExiste = True
    except OSError:
        archivoExiste = False

    if not archivoExiste:
        try:
            with open("entregas.csv", "wt", encoding="utf-8") as arch:
                arch.write("numero_grupo;nombre_proyecto;integrante_responsable;porcentaje_avance\n")
            print("Archivo entregas.csv creado con su encabezado.")
        except OSError as err:
            print("Error al crear archivo:", err)
    else:
        print("El archivo entregas.csv ya existe. Se mantienen los registros previos.")


# Comprobación secuencial de grupo existente propagando errores de apertura
def grupoYaExiste(grupoBuscado):
    encontrado = False
    with open("entregas.csv", "rt", encoding="utf-8") as arch:
        esCabecera = True
        for linea in arch:
            if esCabecera:
                esCabecera = False
            else:
                campos = linea.rstrip("\n").split(";")
                if len(campos) == 4:
                    try:
                        grupoActual = int(campos[0].strip())
                        if grupoActual == grupoBuscado:
                            encontrado = True
                    except ValueError:
                        pass
    return encontrado


# b) Cargar entregas validando que no se repita el grupo
def cargarEntrega():
    try:
        grupo = int(input("Número de grupo: "))
        if grupo <= 0:
            print("El grupo debe ser un valor positivo.")
        else:
            if grupoYaExiste(grupo):
                print(f"El grupo {grupo} ya está cargado.")
            else:
                proyecto = input("Nombre del proyecto: ").strip()
                while proyecto == "" or ";" in proyecto:
                    print("El nombre del proyecto no puede estar vacío ni contener ';'.")
                    proyecto = input("Nombre del proyecto: ").strip()

                responsable = input("Integrante responsable: ").strip()
                while responsable == "" or ";" in responsable:
                    print("El responsable no puede estar vacío ni contener ';'.")
                    responsable = input("Integrante responsable: ").strip()

                avance = float(input("Porcentaje de avance (0 a 100): "))

                if 0 <= avance <= 100 and esAvanceFinito(avance):
                    with open("entregas.csv", "at", encoding="utf-8") as arch:
                        arch.write(f"{grupo};{proyecto};{responsable};{avance}\n")
                    print("Entrega guardada.")
                else:
                    print("El avance debe estar entre 0 y 100. No se guardó el registro.")
    except ValueError:
        print("Error: formato de número no válido.")
    except OSError as err:
        print("Error de archivo (no se pudo verificar o guardar la entrega):", err)


# c y g) Mostrar registros válidos y omitir mal formados
def mostrarEntregas():
    print("Listado de entregas:")
    try:
        with open("entregas.csv", "rt", encoding="utf-8") as arch:
            esCabecera = True
            for linea in arch:
                if esCabecera:
                    esCabecera = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) == 4:
                        try:
                            grupo = int(campos[0].strip())
                            proy = campos[1].strip()
                            resp = campos[2].strip()
                            avance = float(campos[3].strip())
                            if grupo > 0 and proy != "" and resp != "" and 0 <= avance <= 100 and esAvanceFinito(avance):
                                print(f"Grupo {grupo} - {proy} - Responsable: {resp} - Avance: {avance}%")
                            else:
                                print("Registro ignorado por inconsistencias en los datos.")
                        except ValueError:
                            print("Registro ignorado por error de conversión.")
                    else:
                        print("Fila ignorada por cantidad incorrecta de columnas.")
    except OSError as err:
        print("Error al leer:", err)


# d) Búsqueda por número de grupo verificando todos los campos
def buscarEntrega(grupoBuscado):
    print(f"Buscando grupo {grupoBuscado}:")
    hallado = False
    try:
        with open("entregas.csv", "rt", encoding="utf-8") as arch:
            esCabecera = True
            for linea in arch:
                if esCabecera:
                    esCabecera = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) == 4:
                        try:
                            grupo = int(campos[0].strip())
                            proy = campos[1].strip()
                            resp = campos[2].strip()
                            avance = float(campos[3].strip())

                            if grupo > 0 and proy != "" and resp != "" and 0 <= avance <= 100 and esAvanceFinito(avance):
                                if grupo == grupoBuscado:
                                    print(f"Encontrado: {proy} - Responsable: {resp} - Avance: {avance}%")
                                    hallado = True
                        except ValueError:
                            pass
        if not hallado:
            print("Grupo no encontrado.")
    except OSError as err:
        print("Error al buscar:", err)


# e) Proyectos con avance menor al 50% verificando todos los campos
def proyectosAvanceMenor50():
    print("Proyectos con avance menor al 50%:")
    try:
        with open("entregas.csv", "rt", encoding="utf-8") as arch:
            esCabecera = True
            for linea in arch:
                if esCabecera:
                    esCabecera = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) == 4:
                        try:
                            grupo = int(campos[0].strip())
                            proy = campos[1].strip()
                            resp = campos[2].strip()
                            avance = float(campos[3].strip())

                            if grupo > 0 and proy != "" and resp != "" and 0 <= avance <= 100 and esAvanceFinito(avance):
                                if avance < 50:
                                    print(f"Proyecto: {proy} (Grupo {grupo}) - Avance: {avance}%")
                        except ValueError:
                            pass
    except OSError as err:
        print("Error al filtrar proyectos:", err)


# f) Calcular promedio general verificando todos los campos
def calcularPromedioAvance():
    suma = 0.0
    cant = 0
    try:
        with open("entregas.csv", "rt", encoding="utf-8") as arch:
            esCabecera = True
            for linea in arch:
                if esCabecera:
                    esCabecera = False
                else:
                    campos = linea.rstrip("\n").split(";")
                    if len(campos) == 4:
                        try:
                            grupo = int(campos[0].strip())
                            proy = campos[1].strip()
                            resp = campos[2].strip()
                            avance = float(campos[3].strip())

                            if grupo > 0 and proy != "" and resp != "" and 0 <= avance <= 100 and esAvanceFinito(avance):
                                suma += avance
                                cant += 1
                        except ValueError:
                            pass

        if cant > 0:
            print(f"Promedio general de avance: {round(suma / cant, 2)}% (sobre {cant} entregas)")
        else:
            print("No hay registros válidos para calcular el promedio.")

    except OSError as err:
        print("Error al calcular promedio:", err)


def main():
    crearEntregasCsv()

    seguir = "s"
    while seguir == "s":
        cargarEntrega()
        seguir = input("¿Cargar otra entrega? (s/n): ").strip().lower()

    mostrarEntregas()
    buscarEntrega(1)
    proyectosAvanceMenor50()
    calcularPromedioAvance()


main()

Archivo entregas.csv creado con su encabezado.
Entrega guardada.
El grupo 1 ya está cargado.
Entrega guardada.
Listado de entregas:
Grupo 1 - Torneo Esports - Responsable: Uriel Aguilera - Avance: 85.0%
Grupo 2 - Carlos Lujan - Responsable: Carlos Lujan de Monterrey - Avance: 35.0%
Buscando grupo 1:
Encontrado: Torneo Esports - Responsable: Uriel Aguilera - Avance: 85.0%
Proyectos con avance menor al 50%:
Proyecto: Carlos Lujan (Grupo 2) - Avance: 35.0%
Promedio general de avance: 60.0% (sobre 2 entregas)
